# 05 · Contextual aggregator: one final answer from the five agents

Runtime > Change runtime type > **T4 GPU**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

Mounted at /content/drive
Fri Nov 27 18:18:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
import getpass, os
GITHUB_USER = 'natitedros'
REPO = 'CMVA-context-based-aggregation'
token = getpass.getpass('GitHub token (leave blank if repo is public): ')
auth = f'{token}@' if token else ''
%cd /content
!rm -rf /content/repo
!git clone -q https://{auth}github.com/{GITHUB_USER}/{REPO}.git /content/repo
%cd /content/repo
del token, auth

GitHub token (leave blank if repo is public): ··········
/content
/content/repo


In [3]:
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import os
os.environ['PROJECT_ROOT'] = '/content/drive/MyDrive/NLP-Final'
os.environ['SCRATCH_DIR'] = '/content/scratch'

### 1. Build the data and let all ten agents answer the training questions
About an hour. Resumable: if Colab disconnects, rerun the setup cells and this cell; finished agents are skipped.

In [ ]:
!python src/prepare_aggregator_data.py

Loaded 113,828 rows. Columns: ['chatbot', 'Q', 'A', 'rot', 'moral', 'moral-vector', 'A_agrees', 'rot-agree', 'violation-severity', 'worker_answer', 'comments', 'QA', 'split']
Dropping 2 malformed rows (non-numeric A_agrees).
A_agrees values: {2: 66261, 0: 23166, 1: 18766}
split values: {'train': 86529, 'dev': 10842, 'test': 10817}
Sample 'moral' values: ['loyalty', 'care|fairness', 'care', 'authority|sanctity', 'authority']
Ignoring 5 rows with an unrecognised split value.
Using MIC's own train/dev/test split.
{
  "train": 1000,
  "test": 407,
  "test_questions_without_reference": 93,
  "train_profile_share": {
    "care": 0.63,
    "fairness": 0.204,
    "loyalty": 0.229,
    "authority": 0.187,
    "sanctity": 0.106
  },
  "train_values_per_profile": 1.36
}
full_care: answering 1000 training questions
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:15<00:00, 19.06it/s]
  generated 16/1000
  generated 32/1000
  generated 48/1000
  gen

### 2. Train one aggregator per arm (identical settings)

In [ ]:
!python src/train_aggregator.py --arm lora
!python src/train_aggregator.py --arm full

1000 training examples (0 skipped as too long); median length 769 tokens, max 910
Loading weights: 100% 290/290 [00:00<00:00, 323.05it/s]
  0% 0/126 [00:00<?, ?it/s][W1127 02:42:57.080666247 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 998244352 bytes (free: 844955648, total: 15637086208).
  1% 1/126 [00:04<09:20,  4.49s/it][W1127 02:43:00.711402267 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1080033280 bytes (free: 415039488, total: 15637086208).
  5% 6/126 [00:23<07:40,  3.84s/it][W1127 02:43:20.196071570 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1103101952 bytes (free: 521994240, total: 15637086208).
{'loss': '2.295', 'grad_norm': '2.104', 'learning_rate': '0.0001918', 'epoch': '0.16'}
{'loss': '1.804', 'grad_norm': '2.186', 'learning_rate': '0.0001754', 'epoch': '0.32'}
{'loss': '1.839', 'grad_norm': '1.859', 

### 3. Evaluate every system on the held-out questions
The base model and the selection rule need no generation; the prompted and trained aggregators do.

In [ ]:
!python src/eval_aggregator.py --system pt
for arm in ['full', 'lora']:
    !python src/eval_aggregator.py --system select --arm {arm}
    !python src/eval_aggregator.py --system prompted --arm {arm}
    !python src/eval_aggregator.py --system agg --arm {arm}

Loading weights: 100% 201/201 [00:11<00:00, 17.13it/s]
Loading weights: 100% 201/201 [00:04<00:00, 43.21it/s]
Loading weights: 100% 201/201 [00:05<00:00, 35.22it/s]
Loading weights: 100% 201/201 [00:03<00:00, 52.55it/s]
Loading weights: 100% 201/201 [00:06<00:00, 32.53it/s]
{
  "system": "pt",
  "n": 407,
  "rouge1": 0.1513,
  "rouge2": 0.0262,
  "rougeL": 0.1015,
  "rougeLsum": 0.1162,
  "profile_reward": 0.8145,
  "expected_reward": {
    "care": 0.8728,
    "fairness": 0.656,
    "loyalty": 0.7362,
    "authority": 0.7815,
    "sanctity": 0.7453
  }
}
Loading weights: 100% 201/201 [00:00<00:00, 1038.56it/s]
Loading weights: 100% 201/201 [00:00<00:00, 4397.35it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2591.66it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2763.15it/s]
Loading weights: 100% 201/201 [00:00<00:00, 2628.34it/s]
{
  "system": "select_full",
  "n": 407,
  "rouge1": 0.1393,
  "rouge2": 0.0213,
  "rougeL": 0.0915,
  "rougeLsum": 0.1052,
  "profile_reward": 0.8468,
  

### 4. Compare

In [ ]:
import json, pandas as pd
R = os.environ['PROJECT_ROOT'] + '/results'
systems = ['pt', 'select_full', 'select_lora', 'prompted_full', 'prompted_lora', 'agg_full', 'agg_lora']
rows = {s: json.load(open(f'{R}/final_{s}.json')) for s in systems}
pd.DataFrame({s: {k: r[k] for k in ['n', 'rouge1', 'rouge2', 'rougeL', 'rougeLsum', 'profile_reward']} for s, r in rows.items()}).T

,n,rouge1,rouge2,rougeL,rougeLsum,profile_reward
pt,407.0,0.1513,0.0262,0.1015,0.1162,0.8145
select_full,407.0,0.1393,0.0213,0.0915,0.1052,0.8468
select_lora,407.0,0.1393,0.0212,0.0919,0.1051,0.8405
prompted_full,407.0,0.1102,0.0127,0.0772,0.0804,0.8897
prompted_lora,407.0,0.1061,0.0100,0.0741,0.0773,0.9011
agg_full,407.0,0.2684,0.1045,0.2332,0.2333,0.6807
agg_lora,407.0,0.2766,0.1141,0.2390,0.2391,0.6712


In [ ]:
# A few final answers side by side
pd.set_option('display.max_colwidth', 250)
for s in ['pt', 'agg_lora']:
    print(s, pd.read_csv(f'{R}/final_{s}.csv')['reply'].head(3).tolist(), '\n')
print('reference', pd.read_csv(f'{R}/final_pt.csv')['reference'].head(3).tolist())

pt ['As an AI language model, I don\'t have personal relationships or emotions like humans do. However, if "Friend" is referring to someone in a group or community, it could be that they are a member of that group and want to maintain their connection with others.\n\nIf "Friend" is not part of the group, then there might be other reasons why they are not included. For example, they may not feel comfortable being a part of a group or prefer to keep things private. Alternatively, they may simply not know how to join a group or find one that fits their interests.\n\nIn either case, it\'s important for them to', "North Korea's desire to attack the United States is rooted in complex geopolitical and ideological factors that have been simmering for decades. Here are some key reasons why:\n\n1. **National Security Concerns**: North Korea has long been concerned about its neighbors' security and stability. The Korean Peninsula is an important strategic location with significant military and ec

Checking which differences are real:

In [ ]:
import numpy as np, pandas as pd
R = os.environ['PROJECT_ROOT'] + '/results'
load = lambda s: pd.read_csv(f'{R}/final_{s}.csv')
rng = np.random.default_rng(0)

def compare(a, b, col):
    d = (load(a)[col] - load(b)[col]).to_numpy()
    boots = rng.choice(d, (2000, len(d))).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f'{col:15s} {a:14s} − {b:14s} {d.mean():+.3f}   95% CI {lo:+.3f} to {hi:+.3f}')

for col in ['rougeL', 'profile_reward']:
    for a, b in [('agg_lora', 'agg_full'), ('prompted_lora', 'prompted_full'), ('select_lora', 'select_full'),
                 ('agg_lora', 'pt'), ('prompted_lora', 'pt'), ('agg_lora', 'prompted_lora')]:
        compare(a, b, col)
    print()

rougeL          agg_lora       − agg_full       +0.006   95% CI -0.005 to +0.016
rougeL          prompted_lora  − prompted_full  -0.003   95% CI -0.007 to +0.001
rougeL          select_lora    − select_full    +0.000   95% CI -0.002 to +0.003
rougeL          agg_lora       − pt             +0.138   95% CI +0.124 to +0.151
rougeL          prompted_lora  − pt             -0.027   95% CI -0.032 to -0.023
rougeL          agg_lora       − prompted_lora  +0.165   95% CI +0.151 to +0.179

profile_reward  agg_lora       − agg_full       -0.010   95% CI -0.027 to +0.008
profile_reward  prompted_lora  − prompted_full  +0.011   95% CI +0.001 to +0.022
profile_reward  select_lora    − select_full    -0.006   95% CI -0.015 to +0.003
profile_reward  agg_lora       − pt             -0.143   95% CI -0.165 to -0.122
profile_reward  prompted_lora  − pt             +0.087   95% CI +0.073 to +0.100
profile_reward  agg_lora       − prompted_lora  -0.230   95% CI -0.251 to -0.208



Comaring with extra aggregator with 3 epochs and rank 64

In [5]:
!python src/train_aggregator.py --arm lora --epochs 3 --rank 64 --tag _r64e3
!python src/eval_aggregator.py --system agg --arm lora --adapter "$PROJECT_ROOT/models/aggregator/agg_lora_r64e3" --tag _r64e3

config.json: 100% 659/659 [00:00<00:00, 3.52MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 26.6MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 118MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 123MB/s]
1000 training examples (0 skipped as too long); median length 769 tokens, max 910

model.safetensors: downloading bytes:   2% 23.2M/988M [00:01<00:51, 18.7MB/s]
model.safetensors: downloading bytes:  13% 128M/988M [00:01<00:06, 133MB/s, 8.54MB/s  ]  
model.safetensors: downloading bytes:  29% 285M/988M [00:02<00:02, 242MB/s, 22.6MB/s  ]
model.safetensors: downloading bytes:  86% 845M/988M [00:04<00:00, 230MB/s, 66.4MB/s  ]
model.safetensors: reconstructing file:  27% 268M/988M [00:06<00:22, 32.6MB/s, 18.3MB/s  ]
model.safetensors: downloading bytes: 100% 854M/854M [00:11<00:00, 76.2MB/s, 67.7MB/s  ]
model.safetensors: reconstructing file: 100% 988M/988M [00:11<00:00, 88.2MB/s, 70.6MB/s  ]
Loading weights: 100% 290/29

Agent rank sweep

In [ ]:
D = os.environ['PROJECT_ROOT'] + '/data/agent_data_distilled'
A = os.environ['PROJECT_ROOT'] + '/models/agents'
for r in [4, 64]:
    for v in ['care', 'fairness', 'loyalty', 'authority', 'sanctity']:
        !python src/train_agent.py --mode lora --value {v} --data_dir "{D}" --rank {r} --tag _sd_r{r}
        !python src/generate_and_score.py --tag lora_{v}_sd_r{r} --adapter "{A}/lora_{v}_sd_r{r}" --n 500

2000 training examples for lora_care_sd_r4
Loading weights: 100% 290/290 [00:00<00:00, 300.86it/s]
Trainable parameters: 2,199,552 of 496,232,320 (0.44%)
{'loss': '1.175', 'grad_norm': '0.7342', 'learning_rate': '0.0001992', 'epoch': '0.08'}
{'loss': '1.163', 'grad_norm': '0.7204', 'learning_rate': '0.0001909', 'epoch': '0.16'}
{'loss': '1.12', 'grad_norm': '0.6947', 'learning_rate': '0.0001826', 'epoch': '0.24'}
{'loss': '1.105', 'grad_norm': '0.6388', 'learning_rate': '0.0001744', 'epoch': '0.32'}
{'loss': '1.112', 'grad_norm': '0.6258', 'learning_rate': '0.0001661', 'epoch': '0.4'}
{'loss': '1.078', 'grad_norm': '0.6056', 'learning_rate': '0.0001579', 'epoch': '0.48'}
{'loss': '1.101', 'grad_norm': '0.6304', 'learning_rate': '0.0001496', 'epoch': '0.56'}
{'loss': '1.091', 'grad_norm': '0.6585', 'learning_rate': '0.0001413', 'epoch': '0.64'}
{'loss': '1.093', 'grad_norm': '0.6333', 'learning_rate': '0.0001331', 'epoch': '0.72'}
{'loss': '1.072', 'grad_norm': '0.576', 'learning_rate':